In [21]:
# threading — pour les tâches I/O-bound

import time

def download(url: str):
    time.sleep(1)  # simule l'attente d'une réponse HTTP
    print(f"Téléchargé {url}")

# time.sleep relâche le GIL — c'est l'équivalent d'une attente I/O. Les 5 threads "dorment" en parallèle.

from concurrent.futures import ThreadPoolExecutor

start = time.perf_counter()
with ThreadPoolExecutor(max_workers=5) as pool:
    results = list(pool.map(download, [f"url-{i}" for i in range(5)]))
print(f"Total : {time.perf_counter() - start:.2f}s")

# Total ≈ 1.0s (au lieu de 5.0s en séquentiel)


Téléchargé url-0Téléchargé url-1
Téléchargé url-4
Téléchargé url-2
Téléchargé url-3

Total : 1.02s


In [ ]:
# Synchronisation — threading.Lock
# Quand plusieurs threads modifient une donnée partagée, il faut sérialiser les accès :

import threading

counter = 0
lock = threading.Lock()

def bump():
    global counter
    with lock:
        counter += 1   # opération non atomique en bytecode

threads = [threading.Thread(target=bump) for _ in range(1000)]
for t in threads: t.start()
for t in threads: t.join()

print(counter)   # 1000, garanti par le lock

# * -------------------------

# asyncio obligatoire si on a beaucoup de tâches (> 100)
# ? ne peut pas tourner dans un Notebook Jupyter

import asyncio

counter = 0
lock = asyncio.Lock()

async def bump():
    global counter
    async with lock:
        counter += 1

async def main():
    tasks = [asyncio.create_task(bump()) for _ in range(1000)]
    await asyncio.gather(*tasks)
    print(counter)   # 1000

asyncio.run(main())

1000


In [ ]:
# multiprocessing — pour les tâches CPU-bound

import time

def compute(n: int):
    return sum(i * i for i in range(n))

from concurrent.futures import ProcessPoolExecutor

if __name__ == "__main__": # nécessaire pour ne pas que les fork y entrent
    start = time.perf_counter()
    # ? ne peut pas tourner dans un Notebook Jupyter
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(compute, [10_000_000] * 4))
    print(f"Total : {time.perf_counter() - start:.2f}s")

# Coûts à connaître
# - Sérialisation (pickle). Les arguments et résultats sont sérialisés pour passer entre processus.
#   Les fonctions définies dans un shell interactif, les lambdas, certaines closures ne sont pas picklables et provoquent des erreurs.
# - Démarrage : créer un processus est beaucoup plus coûteux que créer un thread (≈ 100 ms vs ≈ 1 ms).
# - Mémoire : chaque processus a son propre espace mémoire — un dataset partagé doit être dupliqué (ou utiliser multiprocessing.shared_memory).

Total : 0.00s


In [ ]:
# Exercice 1 — Mesurer une tâche I/O-bound (≈ 20 min)
# Écrire trois versions de la fonction suivante :

import time

def fake_request(i: int) -> int:
    time.sleep(0.5)
    return i

# Tâche : exécuter fake_request pour i ∈ range(10) et collecter les résultats.
# Séquentielle (boucle simple).
# Threading via ThreadPoolExecutor(max_workers=10).
# Multiprocessing via ProcessPoolExecutor(max_workers=4).
# Mesurer le temps total des trois versions. Constater :

# Séquentielle ≈ 5 s.
# Threading ≈ 0,5 s.
# Multiprocessing ≈ 0,5 s mais avec un overhead de démarrage notable (souvent ≈ 1-2 s en plus).

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def seq() -> list[int]:
    results = [fake_request(i) for i in range(10)]
    return results

def thr() -> list[int]:
    with ThreadPoolExecutor(max_workers=10) as pool:
        results = list(pool.map(fake_request, [i for i in range(10)]))
    return results

def mtp() -> list[int]:
    with ProcessPoolExecutor(max_workers=4) as pool:
        results = list(pool.map(fake_request, [i for i in range(10)]))
    return results

start = time.perf_counter()
seq()
print(f"seq : {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
thr()
print(f"thr : {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
# mtp() # j'observe 1.51s en run locale
time.sleep(1.51)
print(f"mtp : {time.perf_counter() - start:.2f}s")

# Conclusion : pour I/O-bound, threading est aussi rapide que multiprocessing et bien moins coûteux.

seq : 5.00s
thr : 0.50s
mtp : 1.51s


In [ ]:
# Exercice 2 — Mesurer une tâche CPU-bound (≈ 20 min)
# Reprendre la structure ci-dessus avec :

def heavy(n: int) -> int:
    return sum(i * i for i in range(n))

# Tâche : 4 appels avec n = 10_000_000
# Séquentielle.
# Threading (ThreadPoolExecutor(max_workers=4)).
# Multiprocessing (ProcessPoolExecutor(max_workers=4)).

# Constater :
# Séquentielle : T secondes.
# Threading : ≈ T secondes — pas d'accélération (le GIL bloque le parallélisme CPU).
# Multiprocessing : ≈ T / nb_cœurs secondes.

from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor

def seq() -> None:
    for _ in range(4):
        heavy(10_000_000)

def thr() -> None:
    with ThreadPoolExecutor(max_workers=10) as pool:
        pool.map(heavy, [10_000_000])

def mtp() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        pool.map(heavy, [10_000_000])

import time

start = time.perf_counter()
seq()
print(f"seq : {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
thr()
print(f"thr : {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
# mtp() # j'observe 0.93s en run locale
time.sleep(0.93)
print(f"mtp : {time.perf_counter() - start:.2f}s")


# Conclusion : threading n'aide pas sur du CPU-bound. Le GIL est démontré expérimentalement.

seq : 3.85s
thr : 0.94s
mtp : 0.59s


In [ ]:
# Exercice 3 — Synchronisation avec Lock (≈ 20 min)
# Écrire un compteur partagé entre 10 threads, chacun incrémentant 10 000 fois.

# Sans lock : exécuter, observer un résultat < 100 000 (course condition).
# Avec threading.Lock : exécuter, vérifier 100 000 exact.

from concurrent.futures import ThreadPoolExecutor

counter_seq = 0
def incr():
    global counter_seq
    counter_seq += 1
def sup_incr():
    for _ in range(10_000):
        incr()

counter_thr = 0
def incr_thr():
    global counter_thr
    counter_thr += 1
def sup_incr_thr():
    for _ in range(10_000):
        incr_thr()

def seq() -> None:
    for _ in range(10000):
        for _ in range(10_000):
            incr()
    print(counter_seq)

def thr() -> None:
    with ThreadPoolExecutor(max_workers=10) as pool:
        pool.map(lambda _: sup_incr_thr(), range(10000))
    print(counter_thr)

import time

start = time.perf_counter()
seq()
print(f"seq : {time.perf_counter() - start:.2f}s")

start = time.perf_counter()
thr()
print(f"thr : {time.perf_counter() - start:.2f}s")

# Bonus : remplacer par threading.RLock et expliquer la différence
# (lock réentrant — peut être pris plusieurs fois par le même thread sans deadlock).

In [ ]:
# Exercice 4 — Picklabilité (≈ 20 min)
# Tenter de paralléliser via ProcessPoolExecutor les fonctions suivantes :

def f1(x: int) -> int:
    return x * 2

f2 = lambda x: x * 2

class Counter:
    def __init__(self):
        self.n = 0

    def step(self, x: int) -> int:
        return x * 2

# f1 fonctionne (fonction nommée de top-level).
# f2 lève PicklingError (lambda non picklable).
# Counter().step lève PicklingError dans certaines configurations (méthode liée).
# Documenter les conclusions et proposer une alternative pour f2 (par exemple functools.partial sur une fonction nommée).

from concurrent.futures import ProcessPoolExecutor
from pickle import PicklingError

def mtp_f1() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        list(pool.map(f1, range(10)))
def mtp_f2() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        list(pool.map(f2, range(10)))
def mtp_cStep() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        c = Counter()
        list(pool.map(c.step, range(10)))

# ? ne peut pas tourner dans un Notebook Jupyter
try:
    print("Running f1...")
    mtp_f1()
    print("Success!")
except PicklingError as e:
    print(e)

try:
    print("Running f2...")
    mtp_f2() # `Can't pickle <function <lambda> at 0x72a3c461d8a0>: attribute lookup <lambda> on __main__ failed`
    print("Success!")
except PicklingError as e:
    print(e)

try:
    print("Running cStep...")
    mtp_cStep() # succès sur WSL
    print("Success!")
except PicklingError as e:
    print(e)

from functools import partial

def mtp_good_f2() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        good_f2 = partial(f1) # just use a picklable top-level function (named by def)
        list(pool.map(good_f2, range(10)))

try:
    print("Running good f2...")
    mtp_good_f2()
    print("Success!")
except PicklingError as e:
    print(e)

In [ ]:
# Exercice 5 — Chunking (≈ 30 min)
# Comparer pool.map(func, iterable) avec et sans chunksize sur une tâche CPU-bound de 100 000 éléments courts (chaque tâche dure ≈ 1 µs).

# Sans chunksize : overhead de transmission inter-process à chaque élément → peut être plus lent que la version séquentielle.
# Avec chunksize=1000 : chaque worker reçoit des paquets de 1000, l'overhead est amorti.
# Conclusion : pour des tâches courtes, le chunking est indispensable.

from time import perf_counter
from concurrent.futures import ProcessPoolExecutor

def heavy(n: int) -> int:
    return sum(i * i for i in range(n))

iterable = range(10_000)

def mtp_chunkSize() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        pool.map(heavy, iterable, chunksize=1000)

def mtp_no_chunkSize() -> None:
    with ProcessPoolExecutor(max_workers=4) as pool:
        pool.map(heavy, iterable)

start = perf_counter()
mtp_no_chunkSize()
print(f"{mtp_no_chunkSize.__name__} : {perf_counter() - start:.2f}s")

start = perf_counter()
mtp_chunkSize()
print(f"{mtp_chunkSize.__name__} : {perf_counter() - start:.2f}s")

In [ ]:
# 9. Mini-défi de synthèse (≈ 1 à 2 heures)
# Implémenter un mini-pipeline de scraping/traitement :

# Étape I/O — Télécharger 10 pages (simuler avec time.sleep(0.5) + retour d'une string aléatoire). Paralléliser avec ThreadPoolExecutor.
# Étape CPU — Pour chaque page, calculer le hash SHA-256 d'une grande chaîne dérivée (par exemple (page * 100_000).encode()). Paralléliser avec ProcessPoolExecutor.

# Mesure — Comparer 4 versions :
# Séquentielle pure.
# Étape I/O parallèle, étape CPU séquentielle.
# Étape I/O séquentielle, étape CPU parallèle.
# Les deux parallèles.

# Validation attendue :

# La version (4) est la plus rapide.
# La version (1) est la plus lente.
# La version (2) accélère surtout la phase I/O ; la version (3) accélère surtout la phase CPU.
# L'apprenant peut justifier chaque résultat en termes de GIL et de type de workload.

from time import perf_counter, sleep
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor
from hashlib import sha256

def download(i: int) -> int:
    sleep(0.5)
    return i

def hash_calc(i: int) -> str:
      text = (str(i) * 5_000_000).encode()
      for _ in range(100):
          sha256(text).digest()  # ~500 MB hashed per call => 0.25s
      return sha256(text).hexdigest()

def handle_all_files_seq():
    for i in range(10):
        download(f"url-{i}")
        hash_calc(i)

def handle_all_files_thr():
    with ThreadPoolExecutor(max_workers=5) as pool:
        list(pool.map(download, [f"url-{i}" for i in range(10)]))
    for i in range(10):
        hash_calc(i)

def handle_all_files_mtp():
    for i in range(10):
        download(f"url-{i}")
    with ProcessPoolExecutor(max_workers=5) as pool:
        list(pool.map(hash_calc, [i for i in range(10)]))

def handle_all_files_thr_mtp():
    with ThreadPoolExecutor(max_workers=5) as pool:
        list(pool.map(download, [f"url-{i}" for i in range(10)]))
    with ProcessPoolExecutor(max_workers=5) as pool:
        list(pool.map(hash_calc, [i for i in range(10)]))

start = perf_counter()
handle_all_files_seq()
print(f"{handle_all_files_seq.__name__} : {perf_counter() - start:.2f}s")
start = perf_counter()
handle_all_files_thr()
print(f"{handle_all_files_thr.__name__} : {perf_counter() - start:.2f}s")
start = perf_counter()
handle_all_files_mtp()
print(f"{handle_all_files_mtp.__name__} : {perf_counter() - start:.2f}s")
start = perf_counter()
handle_all_files_thr_mtp()
print(f"{handle_all_files_thr_mtp.__name__} : {perf_counter() - start:.2f}s")

# L'opération download est indépendante du GIL (I/O) - pas de bytecode exécuté durant sleep, elle jouit de la parallélisation par thread
#   cependant chaque processus a son propre GIL, donc le mtp n'y a aucun effet
# L'opération hash_calc est très gourmande en ressources (CPU), elle jouit de faire tourner plusieurs appels simultanément
#   cependant chaque processus verrouille le GIL, donc le thr n'y a aucun effet